In [ ]:
pip install gradio_client pillow requests

In [2]:
PERSON_IMAGE  = "imgs/people/m2.jpg"
GARMENT_IMAGE = "imgs/cloths/s1.png"
MASKING_MODE  = "auto"   # auto / upper / lower / full

In [ ]:
# ─────────────────────────────────────────────
# Set your image paths or URLs below
# ─────────────────────────────────────────────

PERSON_IMAGE  = "person.jpg"        # Local path or URL
GARMENT_IMAGE = "garment.jpg"       # Local path or URL
OUTPUT_PATH   = "tryon_result.png"  # Where to save result

# Garment category:
#   "auto"      → Fashn detects automatically Recommended
#   "tops"      → Shirts, jackets, hoodies
#   "bottoms"   → Pants, skirts, shorts
#   "one-piece" → Dresses, jumpsuits, full outfits
CATEGORY = "auto"

# ─────────────────────────────────────────────
# Run the pipeline
# ─────────────────────────────────────────────
try:
    prediction_id = submit_tryon(PERSON_IMAGE, GARMENT_IMAGE, CATEGORY)
    result_url    = poll_status(prediction_id)

    print(f"Result URL: {result_url}")

    download_image(result_url, OUTPUT_PATH)
    print("\nTry-on complete!")

except Exception as e:
    print(f"\nError: {e}")

In [ ]:
# Show result image only
print("📸 Result Image:")
display(IPImage(OUTPUT_PATH))

In [ ]:
# Try multiple garments on the same person

person = "person.jpg"  # Same person for all

garments = [
    {"image": "shirt.jpg",    "category": "tops",      "output": "result_shirt.png"},
    {"image": "pants.jpg",    "category": "bottoms",   "output": "result_pants.png"},
    {"image": "dress.jpg",    "category": "one-piece", "output": "result_dress.png"},
]

results = []

for i, g in enumerate(garments, 1):
    print(f"\n{'='*50}")
    print(f"  Garment {i}/{len(garments)}: {g['image']}")
    print(f"{'='*50}")
    try:
        pred_id    = submit_tryon(person, g["image"], g["category"])
        result_url = poll_status(pred_id)
        saved      = download_image(result_url, g["output"])
        results.append({"garment": g["image"], "output": saved, "status": "✅ Success"})
    except Exception as e:
        results.append({"garment": g["image"], "output": None, "status": f"❌ {e}"})

print("\n\n📋 Batch Summary:")
for r in results:
    print(f"  {r['status']}  {r['garment']}  →  {r['output']}")

In [ ]:
# Display all batch results
for r in results:
    if r["output"]:
        print(f"\n👗 {r['garment']}")
        display(IPImage(r["output"], width=400))

In [3]:
from gradio_client import Client, handle_file
from PIL import Image
import os
from datetime import datetime

# Configuration
HF_TOKEN = ""  # Replace with your token
SPACE_ID = "yisol/IDM-VTON"

print("=" * 50)
print("👔 IDM-VTON Upper Body Virtual Try-On")
print("=" * 50)

# Connect to IDM-VTON
print("\n🔌 Connecting to IDM-VTON...")
client = Client(SPACE_ID, hf_token=HF_TOKEN)
print("✅ Connected successfully!")

# Check available endpoints
print("\n📋 Available API endpoints:")
client.view_api()

👔 IDM-VTON Upper Body Virtual Try-On

🔌 Connecting to IDM-VTON...
Loaded as API: https://yisol-idm-vton.hf.space ✔
✅ Connected successfully!

📋 Available API endpoints:
Client.predict() Usage Info
---------------------------
Named API endpoints: 1

 - predict(dict, garm_img, garment_des, is_checked, is_checked_crop, denoise_steps, seed, api_name="/tryon") -> (output, masked_image_output)
    Parameters:
     - [Imageeditor] dict: Dict(background: filepath | None, layers: List[filepath], composite: filepath | None) (required)  
     - [Image] garm_img: filepath (required)  
     - [Textbox] garment_des: str (required)  
     - [Checkbox] is_checked: bool (not required, defaults to:   True)  
     - [Checkbox] is_checked_crop: bool (not required, defaults to:   False)  
     - [Number] denoise_steps: float (not required, defaults to:   30)  
     - [Number] seed: float (not required, defaults to:   42)  
    Returns:
     - [Image] output: filepath 
     - [Image] masked_image_output: fi

In [ ]:
def simple_tryon(person_img, garment_img, save_as="result.png"):
    """
    Simplest try-on with automatic file handling
    """
    print("🔄 Processing...")
    
    # Run try-on
    temp_result = client.predict(
        dict={"background": handle_file(person_img)},
        garm_img=handle_file(garment_img),
        garment_des="",
        is_checked=True,
        is_checked_crop=False,
        denoise_steps=30,
        seed=42,
        api_name="/tryon"
    )
    
    print(f"📁 Temp result: {temp_result}")
    
    # Try to save - handle all cases
    try:
        if os.path.exists(str(temp_result)):
            # It's a file path
            img = Image.open(str(temp_result))
        elif hasattr(temp_result, 'save'):
            # It's an image object
            img = temp_result
        else:
            # Try to find the file
            import glob
            files = glob.glob("/tmp/gradio/*.png")
            if files:
                latest = max(files, key=os.path.getmtime)
                img = Image.open(latest)
            else:
                raise FileNotFoundError("Cannot find result")
        
        # Save to desired location
        img.save(save_as)
        print(f"✅ Saved to: {save_as}")
        img.show()
        return save_as
        
    except Exception as e:
        print(f"❌ Error saving: {e}")
        # Last resort: check current directory
        import glob
        pngs = glob.glob("*.png")
        print(f"📁 PNGs in current dir: {pngs}")
        
        if pngs:
            latest_png = max(pngs, key=os.path.getmtime)
            print(f"📁 Latest PNG: {latest_png}")
            shutil.copy2(latest_png, save_as)
            print(f"✅ Copied {latest_png} to {save_as}")
            return save_as
        
        return None

# Simple usage
result = simple_tryon(
    person_img="imgs/people/m2.jpg",
    garment_img="imgs/cloths/s1.png",
    save_as="my_tryon_result.png"
)

In [ ]:
from gradio_client import Client, handle_file
from PIL import Image
import matplotlib.pyplot as plt
import os

# Connect
client = Client("yisol/IDM-VTON", hf_token="")

def tryon_display(person_path, garment_path, show_comparison=True):
    """
    Run IDM-VTON and display result in notebook
    """
    
    print(f"🔄 Processing virtual try-on...")
    
    # Run try-on (returns tuple: (result_path, mask_path))
    result = client.predict(
        dict={"background": handle_file(person_path)},
        garm_img=handle_file(garment_path),
        garment_des="",
        is_checked=True,
        is_checked_crop=False,
        denoise_steps=30,
        seed=42,
        api_name="/tryon"
    )
    
    # Extract image paths from tuple
    result_path = result[0]  # First item is the try-on result
    mask_path = result[1]     # Second item is the mask (ignore)
    
    # Load images
    result_img = Image.open(result_path)
    person_img = Image.open(person_path)
    garment_img = Image.open(garment_path)
    
    if show_comparison:
        # Show all three images
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        axes[0].imshow(person_img)
        axes[0].set_title('👤 Original Person', fontsize=14, fontweight='bold')
        axes[0].axis('off')
        
        axes[1].imshow(garment_img)
        axes[1].set_title('👔 Garment', fontsize=14, fontweight='bold')
        axes[1].axis('off')
        
        axes[2].imshow(result_img)
        axes[2].set_title('✨ Try-On Result', fontsize=14, fontweight='bold')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        # Show only result
        plt.figure(figsize=(10, 10))
        plt.imshow(result_img)
        plt.axis('off')
        plt.title('✨ Virtual Try-On Result', fontsize=14, fontweight='bold')
        plt.show()
    
    print(f"✅ Complete! Result size: {result_img.size}")
    return result_img

# 🎯 Use it!
result_img = tryon_display(
    person_path="imgs/people/m2.jpg",
    garment_path="imgs/cloths/s1.png",
    show_comparison=True
)

Loaded as API: https://yisol-idm-vton.hf.space ✔
🔄 Processing virtual try-on...
